# FIAP Global Solution 2026.2 • CitizenTech
## Golpe Zero — Especialização com LoRA (PEFT) vs RAG simples
**Disciplina:** Generative AI for Engineering · **Professor:** Francisco Elânio Bezerra

Este caderno treina um adapter LoRA de verdade sobre o `Qwen2.5-0.5B-Instruct` e compara, nos 35 casos held-out, a especialização paramétrica (LoRA) com a especialização contextual (RAG simples) na mesma base de 0.5B.

1. Fundamentação do LoRA · 2. Dataset de treino · 3. Hiperparâmetros · 4. Treino · 5. Adapter exportado · 6. Placar LoRA vs RAG simples · 7. Conclusão

### 1. Fundamentação do LoRA
Em vez de atualizar a matriz de pesos $W \in \mathbb{R}^{d \times k}$, o LoRA congela $W$ e aprende uma atualização de posto baixo:

$$W' = W + \frac{\alpha}{r} B A, \quad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times k},\; r \ll \min(d, k)$$

Com $r=8$ e $\alpha=16$ aplicados a `q_proj` e `v_proj`, treina-se uma fração mínima dos parâmetros.

### 2. Dataset de treino (sintético, disjunto do gabarito)

In [1]:
import json
from notebooks.dataset_lora import gerar_dataset, verificar_vazamento

dados = gerar_dataset(7)
from collections import Counter
print('exemplos:', len(dados))
print('mensagens de treino idênticas ao gabarito:', verificar_vazamento(dados))
print('distribuição de vereditos:', Counter(json.loads(d['output'])['veredito'] for d in dados))
print('\nExemplo de treino:')
print(dados[0]['input'])
print(dados[0]['output'])

exemplos: 240
mensagens de treino idênticas ao gabarito: 0
distribuição de vereditos: Counter({'golpe': 144, 'nao_consta': 48, 'legitimo': 48})

Exemplo de treino:
A reunião do condomínio será sábado às 10h no salão de festas, tragam sugestões.
{"veredito": "nao_consta", "tipo_golpe": "nao_se_aplica", "sinais_encontrados": ["Assunto sem relação com banco, pagamento ou compra"], "fonte_oficial": "Sistema Golpe Zero", "proximo_passo": "Esta mensagem não trata de golpe financeiro. Pode responder normalmente."}


### 3. Hiperparâmetros LoRA

In [2]:
from peft import LoraConfig, TaskType

cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
                 task_type=TaskType.CAUSAL_LM, target_modules=['q_proj', 'v_proj'])
print(cfg)

W1116 02:35:48.273000 2305 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W1116 02:35:48.314000 2305 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.2', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'q_proj', 'v_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


### 4. Treino real (perda só nos tokens da resposta JSON)

In [3]:
from notebooks.treinar_lora import executar_fine_tuning

resumo = executar_fine_tuning()
print(json.dumps(resumo, indent=2, ensure_ascii=False))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[LoRA] dispositivo=cuda exemplos=240 treinaveis=540,672 de 494,573,440 (0.11%)


[LoRA] epoca 1/3 loss_media=1.5386 (37s)


[LoRA] epoca 2/3 loss_media=0.8227 (75s)


[LoRA] epoca 3/3 loss_media=0.6221 (115s)


[LoRA] adapter salvo em /content/golpe-zero-gs/adapters/golpe_zero_lora
{
  "modelo_base": "Qwen/Qwen2.5-0.5B-Instruct",
  "dispositivo": "cuda",
  "exemplos_treino": 240,
  "epocas": 3,
  "batch": 4,
  "lr": 0.0002,
  "semente": 7,
  "parametros_treinaveis": 540672,
  "parametros_total": 494573440,
  "tempo_treino_seg": 115.6,
  "historico": [
    {
      "epoca": 1,
      "loss_media": 1.5386,
      "loss_final": 1.4064
    },
    {
      "epoca": 2,
      "loss_media": 0.8227,
      "loss_final": 0.5273
    },
    {
      "epoca": 3,
      "loss_media": 0.6221,
      "loss_final": 0.8486
    }
  ]
}


### 5. Adapter exportado

In [4]:
from pathlib import Path

for arq in sorted(Path('adapters/golpe_zero_lora').glob('*')):
    print(f'{arq.name:<30} {arq.stat().st_size/1024:>9.1f} KB')
cfg_salvo = json.load(open('adapters/golpe_zero_lora/adapter_config.json'))
print('base_model_name_or_path:', cfg_salvo['base_model_name_or_path'])

README.md                            0.4 KB
adapter_config.json                  1.1 KB
adapter_model.safetensors         2124.2 KB
base_model_name_or_path: Qwen/Qwen2.5-0.5B-Instruct


### 6. Placar held-out: base zero-shot · base + RAG simples · base + LoRA
Os 35 casos (25 principais + 10 de estresse) não foram vistos no treino.

In [5]:
from avaliacao.benchmark_lora import executar

placar = executar()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[Motor] Carregando modelo de embeddings 'all-MiniLM-L6-v2'...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (112 outputs left out)

  [tc_032] esperado=golpe      obtido=golpe      json=ok 3.5s


  [tc_033] esperado=golpe      obtido=golpe      json=ok 3.4s


  [tc_034] esperado=golpe      obtido=legitimo   json=ok 3.8s


  [tc_035] esperado=golpe      obtido=golpe      json=ok 4.3s

Configuração               |    Acc | Alucin. |   JSON | Fundam. | Lat. média
Base 0.5B Zero-Shot        |  25.7% |   28.6% | 100.0% |    5.3% |      3.15s
Base 0.5B + RAG Simples    |  54.3% |   45.7% | 100.0% |    5.3% |      2.98s
Base 0.5B + LoRA           |  71.4% |   14.3% |  97.1% |   84.2% |      3.57s

Placar salvo em /content/golpe-zero-gs/avaliacao/placar_lora.json


### 7. Conclusão

| Configuração | Acurácia | Alucinação | JSON válido | Fundamentação | Latência média |
| :--- | :---: | :---: | :---: | :---: | :---: |
| Base 0.5B zero-shot | 25.71% | 28.57% | 100.0% | 5.26% | 3.146 s |
| Base 0.5B + RAG simples | 54.29% | 45.71% | 100.0% | 5.26% | 2.984 s |
| Base 0.5B + LoRA | 71.43% | 14.29% | 97.14% | 84.21% | 3.57 s |

Interpretação detalhada em `relatorio/05_lora.md`.